# DIP API Ingestion  Rentenpolitik Monitoring

This notebook queries the German Bundestag's **DIP API** (Dokumentations- und Informationssystem für Parlamentsmaterialien) for legislative procedures (`vorgang`) related to pension policy (*Rentenpolitik*), and writes the raw responses to a local SQLite database and uses dbt to transform

- **Resource chosen: `vorgang`.** Of the 8 DIP resources, `vorgang` ("procedure") is the one that tracks a legislative process end-to-end — from a bill's introduction (`Gesetzentwurf`) through committee stages to passage — including status (`beratungsstand`), topic tags (`sachgebiet`/`deskriptor`), and the responsible chamber. For a Public Affairs early-warning system, this is the single richest entry point: it tells you *what stage something is at*, not just that a document exists. Other resources (`aktivitaet` for individual speeches/questions, `plenarprotokoll-text` for full debate transcripts) would be natural next additions, dip_client.py supports them with the same function call, so extending scope later is not a redesign.
- **Every fetch is logged** to a `fetch_log` table (params, timestamp, item count, error)  so a colleague debugging "why is X missing" can see exactly what was asked for and when, without re-running anything.

**Validation chain applied to every request** (`dip_client.py`, `_request`/`get_resource`):

```
Request
  -> Timeout / network error?   -- yes --> DipTimeoutError / DipTransientError (retried at HTTP layer first)
  -> HTTP status OK (200)?      -- no  --> DipAuthError(401) / DipBadRequestError(400) / DipTransientError(429, 5xx)
  -> Valid JSON?                -- no  --> log raw body, raise DipInvalidResponseError
  -> Expected fields present?   -- no  --> raise DipValidationError
  -> Empty result set?          -- yes --> log info, NOT an error (a real zero-hit query is valid)
  -> Pagination cursor sane?    -- no  --> raise DipApiError (loop guard: repeated-but-unexpected cursor)
  -> Store data
`....

In [2]:
import logging
import os
import sqlite3
from pathlib import Path

from dotenv import load_dotenv

from dip_client import DipClient, fetch_and_store, init_db

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)

load_dotenv()  # reads notebook/.env (see .env.example)
API_KEY = os.environ.get("DIP_API_KEY")

DB_PATH = Path("../dip.sqlite").resolve()  # dbt_project/profiles.yml.example points here
print(f"SQLite DB will be written to: {DB_PATH}")

SQLite DB will be written to: C:\Users\padma\Desktop\CaseStudy\dip.sqlite


In [3]:
conn = sqlite3.connect(DB_PATH)
init_db(conn)

client = DipClient(api_key="R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ")

## Smoke test

Before pulling everything, fetch a handful of items to confirm the key works and the response shape matches what `dip_client.py` expects. If this cell raises `DipApiError`, API key is likely expired/invalid, in such case request a fresh one from `parlamentsdokumentation@bundestag.de` before continuing.**

## Fetch infomration Rentenpolitik-related `vorgang` records



In [4]:
sample = list(client.get_resource("vorgang", params={"f.titel": "Rente"}, max_items=3))
sample[0] if sample else "No results — check filter params against the live Swagger docs."

2026-08-25 14:50:29,297 INFO dip_client: [304e5e4b] GET vorgang params={'f.titel': 'Rente', 'format': 'json'} -> 200 (828 ms)
2026-08-25 14:50:29,305 INFO dip_client: resource=vorgang page=1 items=100 numFound=692


{'id': '337670',
 'beratungsstand': 'Beantwortet',
 'vorgangstyp': 'Schriftliche Frage',
 'typ': 'Vorgang',
 'wahlperiode': 21,
 'aktualisiert': '2026-08-05T14:55:39+02:00',
 'deskriptor': [{'fundstelle': False, 'name': 'Rente', 'typ': 'Sachbegriffe'},
  {'fundstelle': False, 'name': 'Renteneintrittsalter', 'typ': 'Sachbegriffe'},
  {'fundstelle': True, 'name': 'Rentenreform', 'typ': 'Sachbegriffe'}],
 'titel': 'Abschaffung der abschlagsfreien Rente',
 'datum': '2026-07-17'}

In [ ]:
#search terms
QUERIES = [
    {"f.titel": "Rente"},
    {"f.titel": "Rentenversicherung"},
    {"f.titel": "Altersvorsorge"},
    {"f.titel": "Grundrente"},
    {"f.titel": "Rentenreform"},
    # Restrict to the current electoral term to keep the first pull small;
    # remove this filter for a full historical backfill.
    # {"f.wahlperiode": 21},
]

# Each query is run independently and its own success/failure is logged
# and recorded in fetch_log
results = {}
for params in QUERIES:
    term = params["f.titel"]
    try:
        count = fetch_and_store(client, conn, resource="vorgang", params=params)
        results[term] = f"OK ({count} new rows)"
    except Exception as exc:
                results[term] = f"FAILED: {type(exc).__name__}: {exc}"

print("Per-query results:")
for term, outcome in results.items():
    print(f"  {term!r}: {outcome}")

failed = [t for t, o in results.items() if o.startswith("FAILED")]
if failed:
    print(f"\n{len(failed)} of {len(QUERIES)} queries failed -- check fetch_log for details before trusting downstream dbt output.")

2026-08-25 14:54:43,781 INFO dip_client: [805a63d6] GET vorgang params={'f.titel': 'Rente', 'format': 'json'} -> 200 (1078 ms)
2026-08-25 14:54:43,793 INFO dip_client: resource=vorgang page=1 items=100 numFound=692
2026-08-25 14:54:44,373 INFO dip_client: [8a0d1f6f] GET vorgang params={'f.titel': 'Rente', 'format': 'json', 'cursor': 'AoJwkKS-6vgCLlZvcmdhbmctMjc1OTY1'} -> 200 (563 ms)
2026-08-25 14:54:44,380 INFO dip_client: resource=vorgang page=2 items=100 numFound=692
2026-08-25 14:54:47,982 INFO dip_client: [3b2a19f6] GET vorgang params={'f.titel': 'Rente', 'format': 'json', 'cursor': 'AoJwuM6yicwCLVZvcmdhbmctNjY1ODY='} -> 200 (3610 ms)
2026-08-25 14:54:47,984 INFO dip_client: resource=vorgang page=3 items=100 numFound=692
2026-08-25 14:54:49,511 INFO dip_client: [0acd6280] GET vorgang params={'f.titel': 'Rente', 'format': 'json', 'cursor': 'AoJwuJ7YoLQCLVZvcmdhbmctNDA5NjQ='} -> 200 (1515 ms)
2026-08-25 14:54:49,518 INFO dip_client: resource=vorgang page=4 items=100 numFound=692
202

Per-query results:
  'Rente': OK (692 new rows)
  'Rentenversicherung': OK (949 new rows)
  'Altersvorsorge': OK (163 new rows)
  'Grundrente': OK (37 new rows)
  'Rentenreform': OK (25 new rows)


## Sanity checks

In [6]:
import pandas as pd

print("fetch_log:")
display(pd.read_sql("SELECT id, resource, params_json, requested_at, item_count, error FROM fetch_log ORDER BY id", conn))

print("\nraw_documents row count by resource:")
display(pd.read_sql("SELECT resource, COUNT(*) AS n_rows, COUNT(DISTINCT dip_id) AS n_distinct_ids FROM raw_documents GROUP BY resource", conn))

fetch_log:


,id,resource,params_json,requested_at,item_count,error
0,1,vorgang,"{""f.titel"": ""Rente""}",2026-08-23T21:55:50.850208+00:00,692,None
1,2,vorgang,"{""f.titel"": ""Rentenversicherung""}",2026-08-23T21:55:51.619391+00:00,949,None
2,3,vorgang,"{""f.titel"": ""Altersvorsorge""}",2026-08-23T21:55:51.782127+00:00,163,None
3,4,vorgang,"{""f.titel"": ""Grundrente""}",2026-08-23T21:55:51.879346+00:00,37,None
4,5,vorgang,"{""f.titel"": ""Rentenreform""}",2026-08-23T21:55:51.977118+00:00,25,None
5,6,vorgang,"{""f.titel"": ""Rente""}",2026-08-24T07:42:29.703088+00:00,692,None
6,7,vorgang,"{""f.titel"": ""Rentenversicherung""}",2026-08-24T07:42:31.531360+00:00,949,None
7,8,vorgang,"{""f.titel"": ""Altersvorsorge""}",2026-08-24T07:42:32.044957+00:00,163,None
8,9,vorgang,"{""f.titel"": ""Grundrente""}",2026-08-24T07:42:32.213652+00:00,37,None
9,10,vorgang,"{""f.titel"": ""Rentenreform""}",2026-08-24T07:42:32.451165+00:00,25,None



raw_documents row count by resource:


,resource,n_rows,n_distinct_ids
0,vorgang,5598,1767


In [9]:
import json

##  inspect one real record in DB ---
print("\nReal response structure + title field (from fetched data) ===")
row = conn.execute(
    "SELECT payload_json FROM raw_documents WHERE resource='vorgang' LIMIT 1"
).fetchone()
sample = json.loads(row[0])
print("Top-level keys present in a real Vorgang record:")
print(sorted(sample.keys()))
print("\nTitle field ('titel'):", sample.get("titel"))
print("\nFull sample record:")
print(json.dumps(sample, indent=2, ensure_ascii=False))

# fetch vorgangsposition for this Vorgang to find the Drucksache link
vorgang_id = sample["id"]
print(f"\n Vorgang -> Drucksache link (via /vorgangsposition for vorgang id={vorgang_id}) ===")
positions = list(client.get_resource("vorgangsposition", params={"f.vorgang": vorgang_id}, max_items=5))
print(f"Found {len(positions)} vorgangsposition record(s)")
if positions:
    print("\nKeys in a vorgangsposition record:")
    print(sorted(positions[0].keys()))
    print("\nFull sample vorgangsposition record:")
    print(json.dumps(positions[0], indent=2, ensure_ascii=False))
else:
    print("No positions found for this filter -- the filter name 'f.vorgang' may be wrong; "
          "check error output above, or try f.id instead.")


Real response structure + title field (from fetched data) ===
Top-level keys present in a real Vorgang record:
['abstract', 'aktualisiert', 'beratungsstand', 'datum', 'deskriptor', 'id', 'initiative', 'sachgebiet', 'titel', 'typ', 'vorgangstyp', 'wahlperiode']

Title field ('titel'): Haushaltsführung 1998 Überplanmäßige Ausgabe bei Kapitel 1113 Titel 656 03 - Beteiligung des Bundes in knappschaftlicher Rentenversicherung - (G-SIG: 14000059)

Full sample record:
{
  "id": "100154",
  "abstract": "Gewährung eines Zuschusses bis zur Höhe von 279,7 Mio. DM ",
  "beratungsstand": "Abgeschlossen - Ergebnis siehe Vorgangsablauf",
  "vorgangstyp": "Über- und außerplanmäßige Haushaltsausgaben",
  "sachgebiet": [
    "Soziale Sicherung",
    "Öffentliche Finanzen, Steuern und Abgaben"
  ],
  "typ": "Vorgang",
  "wahlperiode": 14,
  "initiative": [
    "Bundesministerium der Finanzen"
  ],
  "aktualisiert": "2022-07-26T19:57:25+02:00",
  "deskriptor": [
    {
      "fundstelle": false,
      "na

2026-08-25 15:06:01,428 INFO dip_client: [eaa4db30] GET vorgangsposition params={'f.vorgang': '100154', 'format': 'json'} -> 200 (500 ms)
2026-08-25 15:06:01,428 INFO dip_client: resource=vorgangsposition page=1 items=4 numFound=4
2026-08-25 15:06:01,609 INFO dip_client: [31b2af7c] GET vorgangsposition params={'f.vorgang': '100154', 'format': 'json', 'cursor': 'AoJcuQI3Vm9yZ2FuZ3Nwb3NpdGlvbi0yMDk2MDU='} -> 200 (188 ms)
2026-08-25 15:06:01,611 INFO dip_client: [31b2af7c] Zero results for this query (numFound=4) -- not an error
2026-08-25 15:06:01,611 INFO dip_client: resource=vorgangsposition page=2 items=0 numFound=4


Found 4 vorgangsposition record(s)

Keys in a vorgangsposition record:
['aktivitaet_anzahl', 'aktualisiert', 'datum', 'dokumentart', 'fortsetzung', 'fundstelle', 'gang', 'id', 'nachtrag', 'titel', 'typ', 'ueberweisung', 'urheber', 'vorgang_id', 'vorgangsposition', 'vorgangstyp', 'zuordnung']

Full sample vorgangsposition record:
{
  "id": "209602",
  "vorgangsposition": "Unterrichtung",
  "zuordnung": "BR",
  "gang": true,
  "fortsetzung": false,
  "nachtrag": false,
  "vorgangstyp": "Über- und außerplanmäßige Haushaltsausgaben",
  "titel": "Haushaltsführung 1998 Überplanmäßige Ausgabe bei Kapitel 1113 Titel 656 03 - Beteiligung des Bundes in knappschaftlicher Rentenversicherung - (G-SIG: 14000059)",
  "typ": "Vorgangsposition",
  "aktivitaet_anzahl": 0,
  "dokumentart": "Drucksache",
  "aktualisiert": "2022-07-26T19:57:15+02:00",
  "urheber": [
    {
      "einbringer": false,
      "bezeichnung": "BMF",
      "titel": "Bundesministerium der Finanzen"
    }
  ],
  "ueberweisung": [
  

In [10]:
# Verify the Drucksache -> Drucksache-Text relationship before building on it
sample_drucksache_id = "82891"  # from your earlier vorgangsposition fetch

dr = list(client.get_resource("drucksache", params={"f.id": sample_drucksache_id}, max_items=1))
print("=== /drucksache response ===")
print(json.dumps(dr[0], indent=2, ensure_ascii=False) if dr else "No result")

dt = list(client.get_resource("drucksache-text", params={"f.id": sample_drucksache_id}, max_items=1))
print("\n=== /drucksache-text response (same id?) ===")
print(json.dumps(dt[0], indent=2, ensure_ascii=False) if dt else "No result -- id may not match, or no text exists for this document")

2026-08-25 15:10:04,931 INFO dip_client: [e27198f8] GET drucksache params={'f.id': '82891', 'format': 'json'} -> 200 (469 ms)
2026-08-25 15:10:04,933 INFO dip_client: resource=drucksache page=1 items=1 numFound=1
2026-08-25 15:10:05,116 INFO dip_client: [582d763a] GET drucksache-text params={'f.id': '82891', 'format': 'json'} -> 200 (188 ms)


=== /drucksache response ===
{
  "id": "82891",
  "drucksachetyp": "Unterrichtung",
  "dokumentart": "Drucksache",
  "autoren_anzahl": 0,
  "typ": "Dokument",
  "vorgangsbezug_anzahl": 1,
  "dokumentnummer": "908/98",
  "wahlperiode": 14,
  "herausgeber": "BR",
  "pdf_hash": "087a7c94c85bb5de691b2049a9a67dc6",
  "aktualisiert": "2022-07-26T19:57:15+02:00",
  "vorgangsbezug": [
    {
      "id": "100154",
      "titel": "Haushaltsführung 1998 Überplanmäßige Ausgabe bei Kapitel 1113 Titel 656 03 - Beteiligung des Bundes in knappschaftlicher Rentenversicherung - (G-SIG: 14000059)",
      "vorgangstyp": "Über- und außerplanmäßige Haushaltsausgaben"
    }
  ],
  "urheber": [
    {
      "einbringer": false,
      "bezeichnung": "BMF",
      "titel": "Bundesministerium der Finanzen"
    }
  ],
  "fundstelle": {
    "pdf_url": "https://dserver.bundestag.de/brd/1998/D908+98.pdf",
    "id": "82891",
    "dokumentnummer": "908/98",
    "datum": "1998-11-13",
    "dokumentart": "Drucksache",
    

2026-08-25 15:10:05,116 INFO dip_client: resource=drucksache-text page=1 items=1 numFound=1



=== /drucksache-text response (same id?) ===
{
  "id": "82891",
  "drucksachetyp": "Unterrichtung",
  "dokumentart": "Drucksache",
  "autoren_anzahl": 0,
  "typ": "Dokument",
  "vorgangsbezug_anzahl": 1,
  "dokumentnummer": "908/98",
  "wahlperiode": 14,
  "herausgeber": "BR",
  "pdf_hash": "087a7c94c85bb5de691b2049a9a67dc6",
  "aktualisiert": "2022-07-26T19:57:15+02:00",
  "vorgangsbezug": [
    {
      "id": "100154",
      "titel": "Haushaltsführung 1998 Überplanmäßige Ausgabe bei Kapitel 1113 Titel 656 03 - Beteiligung des Bundes in knappschaftlicher Rentenversicherung - (G-SIG: 14000059)",
      "vorgangstyp": "Über- und außerplanmäßige Haushaltsausgaben"
    }
  ],
  "urheber": [
    {
      "einbringer": false,
      "bezeichnung": "BMF",
      "titel": "Bundesministerium der Finanzen"
    }
  ],
  "fundstelle": {
    "pdf_url": "https://dserver.bundestag.de/brd/1998/D908+98.pdf",
    "id": "82891",
    "dokumentnummer": "908/98",
    "datum": "1998-11-13",
    "dokumentart": "

In [11]:
conn.close()
print("Done. Point the dbt project's profile at this same SQLite file to build models on top of it.")

Done. Point the dbt project's profile at this same SQLite file to build models on top of it.


In [14]:
import sqlite3
from pathlib import Path

db_path = Path(r"C:\Users\padma\Desktop\CaseStudy\dip.sqlite")
print(db_path.exists())  # confirm before connecting

conn = sqlite3.connect(db_path)

True


In [15]:
# what tables/views exist
cur = conn.execute("SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name")
for row in cur.fetchall():
    print(row)

('fetch_log', 'table')
('mart_rentenpolitik_monitor', 'table')
('raw_documents', 'table')
('sqlite_sequence', 'table')
('stg_dip__vorgaenge', 'view')
('stg_dip__vorgang_initiativen', 'view')
('stg_dip__vorgang_sachgebiete', 'view')


In [16]:
import pandas as pd
df = pd.read_sql("SELECT * FROM mart_rentenpolitik_monitor", conn)
print(df.shape)
df.head(10)

(1767, 13)


,vorgang_id,titel,vorgangstyp,beratungsstand,wahlperiode,datum,aktualisiert,initiativen,sachgebiete,abstract,gesta,dip_url,last_ingested_at
0,337109,Annahmen zur Zuwanderung als Finanzierungsfakt...,Schriftliche Frage,Beantwortet,21,2026-07-03,2026-08-19T11:16:13+02:00,NaN,NaN,NaN,NaN,https://dip.bundestag.de/vorgang/-/337109,2026-08-25T12:54:52.319426+00:00
1,337111,Vorschläge der Rentenkommission zur Zukunft de...,Schriftliche Frage,Beantwortet,21,2026-07-03,2026-08-19T11:16:13+02:00,NaN,NaN,NaN,NaN,https://dip.bundestag.de/vorgang/-/337111,2026-08-25T12:54:52.319426+00:00
2,338145,Durchschnittliche Rente von Rentnern mit minde...,Schriftliche Frage,Beantwortet,21,2026-06-26,2026-08-13T14:59:18+02:00,NaN,NaN,NaN,NaN,https://dip.bundestag.de/vorgang/-/338145,2026-08-25T12:54:52.319426+00:00
3,337240,Maßnahmen zur Behebung der Unterfinanzierung d...,Schriftliche Frage,Beantwortet,21,2026-06-05,2026-08-13T10:33:45+02:00,NaN,NaN,NaN,NaN,https://dip.bundestag.de/vorgang/-/337240,2026-08-25T12:55:14.447805+00:00
4,337281,Kenntnisse über die erwartete Rendite der Alte...,Schriftliche Frage,Beantwortet,21,2026-06-05,2026-08-13T10:33:45+02:00,NaN,NaN,NaN,NaN,https://dip.bundestag.de/vorgang/-/337281,2026-08-25T12:55:14.447805+00:00
5,337009,Kostengünstige private Altersvorsorge sicherst...,Antrag,Abgelehnt,21,2026-07-09,2026-08-06T13:29:56+02:00,Fraktion BÜNDNIS 90/DIE GRÜNEN,Soziale Sicherung,NaN,NaN,https://dip.bundestag.de/vorgang/-/337009,2026-08-25T12:55:16.599093+00:00
6,337668,Anteile von Frauen und Männern an den Zugängen...,Schriftliche Frage,Beantwortet,21,2026-07-17,2026-08-05T14:55:39+02:00,NaN,NaN,NaN,NaN,https://dip.bundestag.de/vorgang/-/337668,2026-08-25T12:54:52.319426+00:00
7,337670,Abschaffung der abschlagsfreien Rente,Schriftliche Frage,Beantwortet,21,2026-07-17,2026-08-05T14:55:39+02:00,NaN,NaN,NaN,NaN,https://dip.bundestag.de/vorgang/-/337670,2026-08-25T12:54:52.319426+00:00
8,333999,Anzahl der Beitragszahler der gesetzlichen Ren...,Schriftliche Frage,Beantwortet,21,2026-03-13,2026-07-28T14:42:14+02:00,NaN,NaN,Originaltext der Frage(n):<br />\r\n<br />\r\n...,NaN,https://dip.bundestag.de/vorgang/-/333999,2026-08-25T12:55:14.447805+00:00
9,333541,Netto-Aufwendungen und Budgetentwicklung der R...,Schriftliche Frage,Beantwortet,21,2026-03-13,2026-07-28T14:31:20+02:00,NaN,NaN,Originaltext der Frage(n):<br />\r\n<br />\r\n...,NaN,https://dip.bundestag.de/vorgang/-/333541,2026-08-25T12:55:14.447805+00:00


In [17]:
print(df.isnull().sum())
print(df["beratungsstand"].value_counts())
if "wahlperiode" in df.columns:
    print(df["wahlperiode"].value_counts().sort_index())

id_col = "id" if "id" in df.columns else "vorgang_id"
print("rows:", len(df), "distinct ids:", df[id_col].nunique())

vorgang_id             0
titel                  0
vorgangstyp            0
beratungsstand        70
wahlperiode            0
datum                  0
aktualisiert           0
initiativen         1078
sachgebiete         1197
abstract             205
gesta               1663
dip_url                0
last_ingested_at       0
dtype: int64
beratungsstand
Beantwortet                                                1209
Abgeschlossen - Ergebnis siehe Vorgangsablauf               174
Abgelehnt                                                   118
Verkündet                                                    62
Nicht abgeschlossen - Einzelheiten siehe Vorgangsablauf      51
Erledigt durch Ablauf der Wahlperiode                        26
Für erledigt erklärt                                         15
Angenommen                                                   14
Abgeschlossen                                                 8
Zusammengeführt mit... (siehe Vorgangsablauf)                 7
Noch ni